# Positional Embedding Analysis: Trained vs Untrained Positions

The N2 model was trained with cluster size 2 (observation_size=2), ground_truth_length=110.
Positions beyond what was seen during training were never updated by gradients — they are still random init.

**Goal:** visualize the embeddings and see if we can spot where the boundary is, without assuming it.

In [ ]:
%matplotlib inline
import torch
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load checkpoint directly to GPU
ckpt_path = '/mnt/model_checkpoints/FixedN/N2/checkpoint_best.pt'
ckpt = torch.load(ckpt_path, map_location=device)
state_dict = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model'].items()}

# Extract positional embeddings: shape (block_size, n_embd) = (1500, 512)
pos_emb = state_dict['transformer.wpe.weight'].float().cpu().numpy()
print(f'Positional embedding shape: {pos_emb.shape}')
print(f'block_size={pos_emb.shape[0]}, n_embd={pos_emb.shape[1]}')

# Part 1: Positional Embedding Boundary Detection

The N2 model has 1500 positional embedding vectors (one per position). We plot the L2 norm of each to find where trained positions end and untrained ones begin.

In [ ]:
norms = np.linalg.norm(pos_emb, axis=1)
variances = np.var(pos_emb, axis=1)

# Detect boundary
window = 30
smoothed_norms = np.convolve(norms, np.ones(window)/window, mode='valid')
norm_diff = np.abs(np.diff(smoothed_norms))
transition_pos = np.argmax(norm_diff) + window // 2

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(norms, linewidth=0.8)
ax.axvline(x=transition_pos, color='red', linestyle='--', alpha=0.7, label=f'Detected boundary ({transition_pos})')
ax.set_xlabel('Position')
ax.set_ylabel('L2 Norm')
ax.set_title('L2 Norm of Positional Embeddings')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Boundary at position {transition_pos}')
print(f'Trained (0-{transition_pos}):   mean norm = {norms[:transition_pos].mean():.4f}')
print(f'Untrained ({transition_pos}-1500): mean norm = {norms[transition_pos:].mean():.4f}')


# Part 2: Representation Collapse in Untrained Traces

Since positional embeddings are ~zero beyond position 369, we expect that all A's (or all G's, etc.) in untrained traces have **identical representations** — the model cannot distinguish between an A at position 400 vs position 600.

We check this by computing pairwise cosine similarity of same-token hidden states:
- **Within each read**: are all A's in trace 5 the same?
- **Across untrained reads**: is an A in trace 5 the same as an A in trace 6?
- **Compared to trained reads**: are A's in traces 1-2 still distinguishable?

In [ ]:
import sys, pickle, random
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from src.data_pkg.IDS_channel import IDS_channel

with open('/workspaces/TReconLM/src/data_pkg/meta_nuc.pkl', 'rb') as f:
    meta = pickle.load(f)
stoi, itos = meta['stoi'], meta['itos']
encode_str = lambda s: [stoi[ch] for ch in s]

channel_stats = {'substitution_probability': 0.02, 'deletion_probability': 0.03, 'insertion_probability': 0.05}

# Load model for representation and attention analysis
from src.gpt_pkg.model import GPT, GPTConfig

config_args = {k: v for k, v in ckpt['model_args'].items() if k != 'model_type'}
model = GPT(GPTConfig(**config_args))
clean_sd = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model'].items()}
model.load_state_dict(clean_sd, strict=True)
model = model.to(device).eval()
print(f'Model loaded on {device}')

rng = random.Random(42)
gt = ''.join(rng.choice('ACGT') for _ in range(110))

def make_input(gt, n_reads, rng):
    reads = [IDS_channel(gt, channel_stats, rng) for _ in range(n_reads)]
    return '|'.join(reads) + ':'

all_cs = list(range(2, 11))
all_inputs = {cs: make_input(gt, cs, random.Random(42)) for cs in all_cs}
for cs in all_cs:
    print(f'CS {cs}: input length = {len(all_inputs[cs])} tokens')


In [ ]:
# Are all A's (or C's, G's, T's) in untrained traces identical?
# Feed CS7 input through the model and extract hidden states at every position after every layer.
# Then check: within untrained reads, do all positions with the same nucleotide have the same representation?

def get_all_hidden_states(model, input_str, device):
    tokens = encode_str(input_str)
    x = torch.tensor([tokens], dtype=torch.long, device=device)
    with torch.no_grad():
        b, t = x.size()
        pos = torch.arange(t, device=device)[None, :].expand(b, -1)
        h = model.transformer.wte(x) + model.transformer.wpe(pos)
        h = model.transformer.drop(h)
        layer_states = [h[0].cpu().numpy()]  # after embedding
        for block in model.transformer.h:
            h, _ = block(h)
            layer_states.append(h[0].cpu().numpy())  # after each layer
    return layer_states  # 13 arrays of shape (T, 512): embedding + 12 layers

states_7 = get_all_hidden_states(model, all_inputs[7], device)
input_tokens_7 = encode_str(all_inputs[7])

# Find read boundaries
sep_positions = [j for j, t in enumerate(input_tokens_7) if t == 5]
read_ranges = []
start = 0
for sp in sep_positions:
    read_ranges.append((start, sp))
    start = sp + 1
read_ranges.append((start, len(input_tokens_7) - 1))

for idx, (rs, re) in enumerate(read_ranges):
    region = "TRAINED" if re < 369 else "UNTRAINED"
    print(f"  Read {idx+1}: pos {rs}-{re} ({region})")

def pairwise_cos(vecs):
    if len(vecs) < 2:
        return np.nan
    nrm = np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-8
    normed = vecs / nrm
    cos = normed @ normed.T
    idx = np.triu_indices(len(vecs), k=1)
    return cos[idx].mean()

# For each nucleotide: across-all-untrained vs across-all-trained, at every layer
token_id_to_name = {0: 'A', 1: 'C', 2: 'G', 3: 'T'}
layer_labels = ['emb'] + [f'L{i}' for i in range(12)]

# Collect data for plotting
plot_data = {}  # {nuc: {'trained': [sim_per_layer], 'untrained': [sim_per_layer]}}

for nuc_id, nuc_name in token_id_to_name.items():
    trained_sims = []
    untrained_sims = []
    
    for layer_idx in range(13):
        h = states_7[layer_idx]
        
        trained_positions = []
        for rs, re in read_ranges:
            if re < 369:
                trained_positions.extend([j for j in range(rs, re) if input_tokens_7[j] == nuc_id])
        
        untrained_positions = []
        for rs, re in read_ranges:
            if rs >= 369:
                untrained_positions.extend([j for j in range(rs, re) if input_tokens_7[j] == nuc_id])
        
        trained_sims.append(pairwise_cos(h[trained_positions]) if len(trained_positions) >= 2 else np.nan)
        untrained_sims.append(pairwise_cos(h[untrained_positions]) if len(untrained_positions) >= 2 else np.nan)
    
    plot_data[nuc_name] = {'trained': trained_sims, 'untrained': untrained_sims}

# Plot: one subplot per nucleotide, x = layer, y = cosine sim
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for idx, (nuc_name, data) in enumerate(plot_data.items()):
    ax = axes[idx]
    ax.plot(range(13), data['trained'], marker='o', linewidth=2, label='Trained reads (1-3)')
    ax.plot(range(13), data['untrained'], marker='s', linewidth=2, color='red', label='Untrained reads (5-7)')
    ax.set_xlabel('Layer (0=embedding)')
    ax.set_ylabel('Pairwise cosine sim')
    ax.set_title(f"Token '{nuc_name}'")
    ax.set_xticks(range(13))
    ax.set_xticklabels(layer_labels, fontsize=7, rotation=45)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Same-token representation similarity across layers\n'
             '(1.0 = all positions with this token are identical = model cannot distinguish them)',
             fontsize=13)
plt.tight_layout()
plt.show()

# Print table
print("\nPairwise cosine similarity: trained vs untrained reads, per nucleotide, per layer")
print(f"{'Layer':>5}", end="")
for nuc in 'ACGT':
    print(f" | {nuc} trained | {nuc} untrained", end="")
print()
print('-' * 85)
for layer_idx in range(13):
    print(f"{layer_labels[layer_idx]:>5}", end="")
    for nuc in 'ACGT':
        t = plot_data[nuc]['trained'][layer_idx]
        u = plot_data[nuc]['untrained'][layer_idx]
        print(f" |    {t:.4f}  |     {u:.4f}   ", end="")
    print()


# Part 3: The `:` token representation across cluster sizes

The `:` token is the generation start point — it appears exactly once per example (always last position).
This makes it the cleanest token to analyze: no within-example ambiguity, just one vector per example.

**Question:** Does the `:` input vector (`token_emb + pos_emb`) look the same for in-distribution
cluster sizes (2–3, where `:` lands on a trained position) vs out-of-distribution ones
(4–10, where `:` lands beyond position ~369)?

In [ ]:


wte = state_dict['transformer.wte.weight'].float().cpu().numpy()
wpe = pos_emb

boundary = 369
n_examples = 100
cluster_sizes = [2, 3, 4, 5, 6, 7, 8, 9, 10]
channel_stats = {'substitution_probability': 0.02, 'deletion_probability': 0.03, 'insertion_probability': 0.05}

def pairwise_cosine_sim(vecs):
    if len(vecs) < 2:
        return np.nan
    nrm = np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-8
    normed = vecs / nrm
    cos = normed @ normed.T
    idx = np.triu_indices(len(vecs), k=1)
    return cos[idx].mean()

colon_id = stoi[':']
colon_data = {cs: {'vecs': [], 'norms': [], 'positions': []} for cs in cluster_sizes}

for cs in cluster_sizes:
    for ex_idx in range(n_examples):
        rng = random.Random(1000 * cs + ex_idx)
        gt = ''.join(rng.choice('ACGT') for _ in range(110))
        reads = [IDS_channel(gt, channel_stats, rng) for _ in range(cs)]
        input_str = '|'.join(reads) + ':'
        tokens = encode_str(input_str)
        colon_pos = len(tokens) - 1
        vec = wte[colon_id] + wpe[colon_pos]
        colon_data[cs]['vecs'].append(vec)
        colon_data[cs]['norms'].append(np.linalg.norm(vec))
        colon_data[cs]['positions'].append(colon_pos)

colon_cos_sims = {}
for cs in cluster_sizes:
    colon_cos_sims[cs] = pairwise_cosine_sim(np.array(colon_data[cs]['vecs']))

print(f"{'CS':>3} | {'Mean pos of :':>13} | {'Norm':>18} | {'Cross-example cos sim':>22}")
print('-' * 70)
for cs in cluster_sizes:
    norms_arr = np.array(colon_data[cs]['norms'])
    pos_arr = np.array(colon_data[cs]['positions'])
    print(f"{cs:>3} | {pos_arr.mean():>10.0f} ± {pos_arr.std():>2.0f} | "
          f"{norms_arr.mean():.4f} ± {norms_arr.std():.4f} | "
          f"{colon_cos_sims[cs]:.6f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

norm_means = [np.mean(colon_data[cs]['norms']) for cs in cluster_sizes]
norm_stds = [np.std(colon_data[cs]['norms']) for cs in cluster_sizes]
ax1.errorbar(cluster_sizes, norm_means, yerr=norm_stds,
            marker='o', linewidth=2, capsize=4, color='tab:blue')
ax1.set_xlabel('Cluster Size')
ax1.set_ylabel('L2 Norm of : input vector')
ax1.set_title("Norm of ':' token representation\n(token_emb + pos_emb)")
ax1.set_xticks(cluster_sizes)
ax1.grid(True, alpha=0.3)

cos_vals = [colon_cos_sims[cs] for cs in cluster_sizes]
ax2.plot(cluster_sizes, cos_vals, marker='s', linewidth=2, color='tab:red')
ax2.set_xlabel('Cluster Size')
ax2.set_ylabel('Cross-Example Pairwise Cosine Sim')
ax2.set_title("':' token similarity across 100 examples\n(1.0 = all identical)")
ax2.set_xticks(cluster_sizes)
ax2.set_ylim(0.5, 1.05)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Part 4: Attention Analysis

## 6. Attention heatmaps: all 12 layers, cluster size 2 vs 7

Head-averaged attention patterns for every layer. CS2 (all positions trained) should show
structured patterns; CS7 (many untrained positions) should show diffuse/broken patterns.

In [ ]:
n_layers = len(model.transformer.h)
n_heads = model.config.n_head
print(f'{n_layers} layers, {n_heads} heads')

# Attention during GENERATION: x-axis = input (prompt), y-axis = generated tokens
# For each cluster size, run actual generation and capture attention at each decoding step

all_cs = list(range(2, 11))
all_inputs = {}
for cs in all_cs:
    all_inputs[cs] = make_input(gt, cs, random.Random(42))

def get_generation_attention(model, input_str, device, max_new_tokens=110):
    """Run autoregressive generation and capture attention at each step.
    Returns attention from each generated token to ALL preceding positions (input + generated so far).
    We extract only the attention to INPUT positions for the plot.
    """
    tokens = encode_str(input_str)
    input_len = len(tokens)
    x = torch.tensor([tokens], dtype=torch.long, device=device)
    
    with torch.no_grad():
        # First: process the full input through all layers to build KV cache
        b, t = x.size()
        pos = torch.arange(t, device=device)[None, :].expand(b, -1)
        h = model.transformer.wte(x) + model.transformer.wpe(pos)
        h = model.transformer.drop(h)
        
        # Run through all layers, storing KV caches
        caches = []
        for block in model.transformer.h:
            h, cache = block(h, store_attention=True)
            caches.append(cache)
        h = model.transformer.ln_f(h)
        
        # Now generate tokens one by one, capturing attention each step
        # gen_attentions[layer] will be (n_heads, n_generated, total_seq_len)
        gen_attentions = [[] for _ in range(len(model.transformer.h))]
        
        for step in range(max_new_tokens):
            # Get logits from last position
            logits = model.lm_head(h[:, -1:, :])  # (1, 1, vocab)
            next_token = logits.argmax(dim=-1)  # greedy
            
            # Get next position
            next_pos = torch.tensor([[input_len + step]], device=device)
            h_next = model.transformer.wte(next_token) + model.transformer.wpe(next_pos)
            h_next = model.transformer.drop(h_next)
            
            # Run through layers with cache, storing attention
            new_caches = []
            for layer_idx, block in enumerate(model.transformer.h):
                h_next, new_cache = block(h_next, cache=caches[layer_idx], store_attention=True)
                new_caches.append(new_cache)
                # Attention shape: (1, n_heads, 1, total_seq_so_far)
                attn_weights = block.attn.last_attention_weights[0, :, 0, :]  # (n_heads, total_seq)
                gen_attentions[layer_idx].append(attn_weights.cpu().numpy())
            caches = new_caches
        
        # Stack: (n_heads, n_generated, total_seq_at_last_step) - but each step has different total_seq
        # We only care about attention to INPUT positions, so take [:input_len] from each
        result = []
        for layer_idx in range(len(model.transformer.h)):
            layer_attn = np.zeros((gen_attentions[layer_idx][0].shape[0], max_new_tokens, input_len))
            for step in range(max_new_tokens):
                a = gen_attentions[layer_idx][step]  # (n_heads, total_seq_so_far)
                layer_attn[:, step, :input_len] = a[:, :input_len]
            result.append(layer_attn)  # (n_heads, n_generated, input_len)
        
    return result

# Get generation attention for all cluster sizes
all_gen_attn = {}
for cs in all_cs:
    all_gen_attn[cs] = get_generation_attention(model, all_inputs[cs], device)
    print(f'CS {cs}: gen attn shape = {all_gen_attn[cs][0].shape} (heads, gen_tokens, input_len)')

# Plot: rows = layers, columns = cluster sizes
# Raw attention values (no normalization) with colorbars showing actual range
fig, axes = plt.subplots(n_layers, len(all_cs), figsize=(3.5 * len(all_cs), 3 * n_layers))

for layer_idx in range(n_layers):
    for col, cs in enumerate(all_cs):
        avg_attn = all_gen_attn[cs][layer_idx].mean(axis=0)  # (n_generated, input_len)
        
        ax = axes[layer_idx, col]
        im = ax.imshow(avg_attn, cmap='viridis', aspect='auto')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        if layer_idx == 0:
            ax.set_title(f'CS {cs}', fontsize=11)
        if col == 0:
            ax.set_ylabel(f'L{layer_idx}', fontsize=10)
        if layer_idx == n_layers - 1:
            ax.set_xlabel('Input pos')
        else:
            ax.set_xticks([])
        if col == 0:
            ax.set_yticks([0, 54, 109])
            ax.set_yticklabels(['0', '55', '110'], fontsize=7)
        else:
            ax.set_yticks([])

plt.suptitle('Generation attention (raw values + colorbars): y = predicted GT token, x = input position\n'
             'Rows = layers (0-11), Columns = cluster sizes (2-10)',
             fontsize=14, y=1.005)
plt.tight_layout()
plt.show()


In [ ]:
# Simple summary: what fraction of generation attention goes to trained vs untrained reads?
# Per cluster size, averaged over all generated tokens, all heads, last layer.

last_layer = n_layers - 1
boundary = 369

print("Fraction of generation attention going to TRAINED input positions (pos < 369)")
print("(Last layer, head-averaged, averaged over all 110 generated tokens)\n")
print(f"{'CS':>3} | {'Input len':>9} | {'To trained':>10} | {'To untrained':>12} | {'To trained reads':>15}")
print('-' * 60)

for cs in all_cs:
    avg_attn = all_gen_attn[cs][last_layer].mean(axis=0)  # (110, input_len)
    input_len = avg_attn.shape[1]
    
    # Average over all generated positions
    mean_attn = avg_attn.mean(axis=0)  # (input_len,)
    
    trained_frac = mean_attn[:min(boundary, input_len)].sum()
    untrained_frac = mean_attn[boundary:].sum() if input_len > boundary else 0
    
    # Also: what fraction goes to the first 2 reads specifically?
    # (the reads the model was trained with)
    input_tokens = encode_str(all_inputs[cs])
    seps = [j for j, t in enumerate(input_tokens) if t == 5]
    if len(seps) >= 2:
        first_2_reads_end = seps[1]  # end of read 2
    else:
        first_2_reads_end = input_len
    first_2_frac = mean_attn[:first_2_reads_end].sum()
    
    print(f"{cs:>3} | {input_len:>9} | {trained_frac:>10.1%} | {untrained_frac:>12.1%} | {first_2_frac:>15.1%}")

print("\nIf the model could ignore untrained traces, 'To untrained' would be ~0%.")
print("Instead, the more traces we add, the more attention leaks to them.")


### Comparison: Model trained on all cluster sizes (obs 2-10)

Same visualization but with the model trained on observation_size=10 (sees all cluster sizes during training).
This model should have trained positional embeddings for much longer sequences, so attention
should remain structured even for large cluster sizes.

In [ ]:
# Load the obs10 model (trained on all cluster sizes)
ckpt_obs10_path = '/mnt/model_checkpoints/Reproduce/ids_data_nuc_CPRED_obs10_gt110_compute_final_110nt_reproduce/final_110nt_reproduce/train_run_gpt_20250806_100623/checkpoint_best.pt'
ckpt_obs10 = torch.load(ckpt_obs10_path, map_location=device)
config_args_obs10 = {k: v for k, v in ckpt_obs10['model_args'].items() if k != 'model_type'}
model_obs10 = GPT(GPTConfig(**config_args_obs10))
clean_sd_obs10 = {k.replace('_orig_mod.', ''): v for k, v in ckpt_obs10['model'].items()}
model_obs10.load_state_dict(clean_sd_obs10, strict=True)
model_obs10 = model_obs10.to(device).eval()
print(f'Obs10 model loaded on {device}')

# Check positional embedding norms for this model
pos_emb_obs10 = clean_sd_obs10['transformer.wpe.weight'].float().cpu().numpy()
norms_obs10 = np.linalg.norm(pos_emb_obs10, axis=1)
print(f'Obs10 pos emb norms: pos 0-100={norms_obs10[:100].mean():.4f}, pos 300-400={norms_obs10[300:400].mean():.4f}, pos 700-800={norms_obs10[700:800].mean():.4f}, pos 1000-1100={norms_obs10[1000:1100].mean():.4f}')

# Get generation attention for all cluster sizes with obs10 model
all_gen_attn_obs10 = {}
for cs in all_cs:
    all_gen_attn_obs10[cs] = get_generation_attention(model_obs10, all_inputs[cs], device)
    print(f'CS {cs}: done')

# Plot 1: N2 model (trained on cluster size 2 only) — all 12 layers
fig, axes = plt.subplots(n_layers, len(all_cs), figsize=(3.5 * len(all_cs), 3 * n_layers))
for layer_idx in range(n_layers):
    for col, cs in enumerate(all_cs):
        avg_attn = all_gen_attn[cs][layer_idx].mean(axis=0)
        ax = axes[layer_idx, col]
        im = ax.imshow(avg_attn, cmap='viridis', aspect='auto')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        if layer_idx == 0:
            ax.set_title(f'CS {cs}', fontsize=11)
        if col == 0:
            ax.set_ylabel(f'L{layer_idx}', fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
plt.suptitle('N2 MODEL (trained on cluster size 2 only): generation attention\n'
             'y = predicted GT token, x = input position',
             fontsize=14, y=1.005)
plt.tight_layout()
plt.show()

# Plot 2: Obs10 model (trained on all cluster sizes) — all 12 layers
fig, axes = plt.subplots(n_layers, len(all_cs), figsize=(3.5 * len(all_cs), 3 * n_layers))
for layer_idx in range(n_layers):
    for col, cs in enumerate(all_cs):
        avg_attn = all_gen_attn_obs10[cs][layer_idx].mean(axis=0)
        ax = axes[layer_idx, col]
        im = ax.imshow(avg_attn, cmap='viridis', aspect='auto')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        if layer_idx == 0:
            ax.set_title(f'CS {cs}', fontsize=11)
        if col == 0:
            ax.set_ylabel(f'L{layer_idx}', fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
plt.suptitle('OBS10 MODEL (trained on all cluster sizes): generation attention\n'
             'y = predicted GT token, x = input position',
             fontsize=14, y=1.005)
plt.tight_layout()
plt.show()

del model_obs10, ckpt_obs10
torch.cuda.empty_cache()
